In [136]:
import pandas as pd
import json
import os
from tqdm import tqdm
tqdm.pandas()

task = 'openset'
final_df = []
for method in os.listdir(f"results/{task}"):
    result_file = f"results/{task}/{method}/results.csv"
    df =pd.read_csv(result_file)
    df = df[~df['args'].isna()]
    df['args'] = df['args'].progress_apply(json.loads)
    df['fold_idx'] = df['args'].apply(lambda x: int(x['fold_idx']))
    df['num_train_epochs'] = df['args'].apply(lambda x: x['num_train_epochs'])
    # df['num_pretrain_epochs'] = df['args'].apply(lambda x: x['num_pretrain_epochs'])
    df = df[(df['num_train_epochs'].apply(int)==50)]
    df = df.drop(['cluster_num_factor', 'args', 'seed', 'num_train_epochs'], axis=1)
    df = df.drop_duplicates(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
    final_df.append(df)

final_df = pd.concat(final_df)
final_df = final_df.set_index(['labeled_ratio', 'known_cls_ratio', 'dataset', 'method', 'fold_idx'])
df_melted = final_df.stack().reset_index()
df_melted = df_melted.rename(columns={0: "value", "level_5": "metric"})
df_melted = df_melted.sort_values(['dataset', 'method', 'fold_idx', 'labeled_ratio', 'known_cls_ratio', 'metric'])
df_melted

100%|██████████| 329/329 [00:00<00:00, 67849.64it/s]


,labeled_ratio,known_cls_ratio,dataset,method,fold_idx,metric,value
500,0.1,0.25,20NG,ADB,0,ACC,53.8000
501,0.1,0.25,20NG,ADB,0,F1,50.7301
502,0.1,0.25,20NG,ADB,0,K-F1,48.3731
503,0.1,0.25,20NG,ADB,0,N-F1,62.5150
440,1.0,0.25,20NG,ADB,0,ACC,64.6900
...,...,...,...,...,...,...,...
879,0.1,0.50,thucnews,clap,0,N-F1,0.0000
904,0.1,0.75,thucnews,clap,0,ACC,50.3200
905,0.1,0.75,thucnews,clap,0,F1,54.4698
906,0.1,0.75,thucnews,clap,0,K-F1,0.0000


In [137]:
df_pivot = df_melted.pivot(index=['dataset', 'method', 'fold_idx'], columns=['labeled_ratio', 'known_cls_ratio', 'metric'], values='value')
df_pivot.to_excel(f'results/{task}_pivot.xlsx')
df_pivot

labeled_ratio                      0.1                                   \
known_cls_ratio                   0.25                                    
metric                             ACC         F1       K-F1       N-F1   
dataset       method  fold_idx                                            
20NG          ADB     0         53.800  50.730100  48.373100  62.515000   
                      3         51.450  56.462700  56.991000  53.821000   
              DOC     0          0.785   0.379602   0.280936   0.872931   
              DeepUnk 0          0.807   0.459434   0.374406   0.884569   
              DyEn    0          0.991   0.984290   0.982358   0.993952   
...                                ...        ...        ...        ...   
stackoverflow ab      0         73.700  58.100000  53.100000  83.200000   
                      3         80.000  65.400000  61.000000  87.000000   
              clap    0         31.110  47.271500   0.000000   0.000000   
thucnews      ab      0         32.700  26.700000  22.600000  43.100000   
              clap    0         22.940  32.559800   0.000000   0.000000   

labeled_ratio                     1.0                                 0.1  \
known_cls_ratio                  0.25                                0.50   
metric                            ACC       F1     K-F1     N-F1      ACC   
dataset       method  fold_idx                                              
20NG          ADB     0         64.69  67.9614  67.5778  69.8795      NaN   
                      3           NaN      NaN      NaN      NaN      NaN   
              DOC     0           NaN      NaN      NaN      NaN   0.5625   
              DeepUnk 0           NaN      NaN      NaN      NaN   0.5885   
              DyEn    0           NaN      NaN      NaN      NaN      NaN   
...                               ...      ...      ...      ...      ...   
stackoverflow ab      0           NaN      NaN      NaN      NaN  66.9000   
                      3           NaN      NaN      NaN      NaN  68.8000   
              clap    0           NaN      NaN      NaN      NaN  43.9700   
thucnews      ab      0           NaN      NaN      NaN      NaN  34.2000   
              clap    0           NaN      NaN      NaN      NaN  35.1900   

labeled_ratio                                                             \
known_cls_ratio                                                     0.75   
metric                                 F1       K-F1       N-F1      ACC   
dataset       method  fold_idx                                             
20NG          ADB     0               NaN        NaN        NaN      NaN   
                      3               NaN        NaN        NaN      NaN   
              DOC     0          0.231086   0.184650   0.695440   0.4450   
              DeepUnk 0          0.330367   0.292578   0.708259   0.5365   
              DyEn    0               NaN        NaN        NaN      NaN   
...                                   ...        ...        ...      ...   
stackoverflow ab      0         63.400000  62.500000  72.100000  61.6000   
                      3         65.300000  64.500000  73.500000      NaN   
              clap    0         59.073600   0.000000   0.000000  80.7700   
thucnews      ab      0         28.300000  26.200000  43.500000  27.1000   
              clap    0         43.534400   0.000000   0.000000  50.3200   

labeled_ratio                                                    1.0           \
known_cls_ratio                                                 0.50            
metric                                 F1       K-F1       N-F1  ACC  F1 K-F1   
dataset       method  fold_idx                                                  
20NG          ADB     0               NaN        NaN        NaN  NaN NaN  NaN   
                      3               NaN        NaN        NaN  NaN NaN  NaN   
              DOC     0          0.337526   0.327652   0.485635  NaN NaN  NaN   
          

In [138]:
# 基于 df_melted = [labeled_ratio, known_cls_ratio, dataset, method, fold_idx, metric, value]

# 认为：只要某个 metric 有记录，就视为该 (dataset, method, labeled_ratio, known_cls_ratio, fold_idx) 已测试
cells = (df_melted[["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"]]
         .drop_duplicates()
         .sort_values(["dataset","method","labeled_ratio","known_cls_ratio","fold_idx"])
         .reset_index(drop=True))

# 1) 每个 dataset × method 的维度覆盖与计数
progress_by_dm = (cells
    .groupby(["dataset","method"])
    .agg(
        labeled_ratios=("labeled_ratio", lambda s: sorted(pd.unique(s))),
        n_labeled=("labeled_ratio", pd.Series.nunique),
        known_cls_ratios=("known_cls_ratio", lambda s: sorted(pd.unique(s))),
        n_known=("known_cls_ratio", pd.Series.nunique),
        folds=("fold_idx", lambda s: sorted(pd.unique(s))),
        n_folds=("fold_idx", pd.Series.nunique),
        n_cells=("fold_idx", "size"),  # 完成的组合总数
    )
    .reset_index()
)

# 2) 每个 dataset × method × labeled_ratio × known_cls_ratio 下完成了多少个 fold
fold_coverage = (cells
    .groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
    .nunique()
    .reset_index(name="folds_done")
    .sort_values(["dataset","method","labeled_ratio","known_cls_ratio"])
)

In [139]:
# ====== 2) （可选）若有计划 fold 列表，显示“完成数/计划数” ======
# 例如计划 folds 为 [0,1,2,3,4]（共5个）
EXPECTED_FOLDS = None  # e.g., [0,1,2,3,4]
if EXPECTED_FOLDS is not None:
    plan_n = len(EXPECTED_FOLDS)
    # 统计每格完成的 fold 个数
    fold_done_cnt = fold_coverage.copy()
    fold_done_cnt["plan"] = plan_n
    fold_done_cnt["text"] = fold_done_cnt["folds_done"].astype(int).astype(str) + "/" + str(plan_n)
    pivot_folds_text = fold_done_cnt.pivot(
        index=["dataset", "method"],
        columns=["labeled_ratio", "known_cls_ratio"],
        values="text"
    ).sort_index()

# ====== 3) 每格显示“已完成的 fold 列表”，便于直观看缺哪几个 ======
# 先做一个按格聚合出 fold 列表的表
fold_list = (cells.groupby(["dataset","method","labeled_ratio","known_cls_ratio"])["fold_idx"]
             .apply(lambda s: sorted(pd.unique(s)))
             .reset_index(name="folds_list"))
fold_list["folds_list_str"] = fold_list["folds_list"].apply(lambda x: "[" + ", ".join(map(str, x)) + "]")

pivot_folds_list = fold_list.pivot(
    index=["dataset", "method"],
    columns=["labeled_ratio", "known_cls_ratio"],
    values="folds_list_str"
).sort_index()

pivot_folds_list.to_excel(f'results/{task}_progress.xlsx')
pivot_folds_list

labeled_ratio             0.1  1.0     0.1       1.0
known_cls_ratio          0.25 0.25    0.50 0.75 0.50
dataset       method                                
20NG          ADB      [0, 3]  [0]     NaN  NaN  NaN
              DOC         [0]  NaN     [0]  [0]  NaN
              DeepUnk     [0]  NaN     [0]  [0]  NaN
              DyEn     [0, 3]  NaN     NaN  NaN  NaN
              KnnCon   [0, 3]  NaN     NaN  NaN  NaN
...                       ...  ...     ...  ...  ...
stackoverflow KnnCon   [0, 3]  NaN     NaN  NaN  NaN
              ab       [0, 3]  NaN  [0, 3]  [0]  NaN
              clap        [0]  NaN     [0]  [0]  NaN
thucnews      ab          [0]  NaN     [0]  [0]  NaN
              clap        [0]  NaN     [0]  [0]  NaN

[76 rows x 5 columns]

In [140]:
all_num = df_melted['dataset'].nunique() * 3 * 3 * df_melted['method'].nunique() * 5
cur_num = len(df_melted) / df_melted['metric'].nunique()
cur_num / all_num

0.05757575757575758

In [141]:
all_num

3960

In [142]:
cur_num

228.0